# YOLO11 Video Segmentation

This notebook demonstrates how to perform object segmentation on videos using YOLO11.
We'll process a prerecorded video frame by frame and output a segmented video.


In [2]:
!pip install tqdm

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [3]:
# Import required libraries
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from pathlib import Path
import time


In [4]:
# Load the YOLO11 segmentation model
model = YOLO("yolo11n-seg.pt")  # You can use yolo11s-seg.pt, yolo11m-seg.pt, etc. for better accuracy
print(f"Model loaded: {model.model_name}")


Model loaded: yolo11n-seg.pt


In [5]:
# Download a sample video for demonstration
import urllib.request

# Sample video URL (you can replace with your own video)
video_url = "https://sample-videos.com/zip/10/mp4/SampleVideo_1280x720_1mb.mp4"
input_video_path = "sample_video.mp4"

# Download video if it doesn't exist
if not os.path.exists(input_video_path):
    print("Downloading sample video...")
    try:
        urllib.request.urlretrieve(video_url, input_video_path)
        print(f"Video downloaded: {input_video_path}")
    except Exception as e:
        print(f"Failed to download video: {e}")
        print("Please provide your own video file and update the input_video_path variable")
        input_video_path = "your_video.mp4"  # Update this with your video path
else:
    print(f"Using existing video: {input_video_path}")


Failed to download video: <urlopen error [Errno 110] Connection timed out>
Please provide your own video file and update the input_video_path variable


In [ ]:
# Video processing configuration
output_video_path = "segmented_output.mp4"
confidence_threshold = 0.5
show_preview = True  # Set to False for faster processing
max_frames = None  # Set to a number to limit processing (e.g., 100 for testing)

print(f"Input video: {input_video_path}")
print(f"Output video: {output_video_path}")
print(f"Confidence threshold: {confidence_threshold}")


In [ ]:
def process_video_with_segmentation(input_path, output_path, model, conf_threshold=0.5, show_preview=False, max_frames=None):
    """
    Process video with YOLO segmentation and save the result.
    
    Args:
        input_path: Path to input video
        output_path: Path to save output video
        model: YOLO model instance
        conf_threshold: Confidence threshold for detections
        show_preview: Whether to show preview during processing
        max_frames: Maximum number of frames to process (None for all)
    """
    
    # Open input video
    cap = cv2.VideoCapture(input_path)
    
    if not cap.isOpened():
        raise ValueError(f"Error opening video file: {input_path}")
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if max_frames:
        total_frames = min(total_frames, max_frames)
    
    print(f"Video properties:")
    print(f"  Resolution: {width}x{height}")
    print(f"  FPS: {fps}")
    print(f"  Total frames to process: {total_frames}")
    
    # Define codec and create VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Initialize tracking variables
    frame_count = 0
    processing_times = []
    
    # Create progress bar
    pbar = tqdm(total=total_frames, desc="Processing video")
    
    try:
        while True:
            ret, frame = cap.read()
            
            if not ret or (max_frames and frame_count >= max_frames):
                break
            
            # Record processing start time
            start_time = time.time()
            
            # Run YOLO segmentation on the frame
            results = model(frame, conf=conf_threshold, verbose=False)
            
            # Get annotated frame with segmentation masks
            annotated_frame = results[0].plot()
            
            # Record processing time
            processing_time = time.time() - start_time
            processing_times.append(processing_time)
            
            # Write frame to output video
            out.write(annotated_frame)
            
            # Show preview if enabled
            if show_preview:
                # Resize for display if too large
                display_frame = annotated_frame.copy()
                if width > 1280:
                    scale = 1280 / width
                    new_width = int(width * scale)
                    new_height = int(height * scale)
                    display_frame = cv2.resize(display_frame, (new_width, new_height))
                
                cv2.imshow('Video Segmentation', display_frame)
                
                # Break on 'q' key press
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    print("\\nProcessing interrupted by user")
                    break
            
            frame_count += 1
            pbar.update(1)
            
            # Update progress bar description with current FPS
            if frame_count % 10 == 0:  # Update every 10 frames
                avg_time = np.mean(processing_times[-10:])
                current_fps = 1.0 / avg_time if avg_time > 0 else 0
                pbar.set_description(f"Processing video (FPS: {current_fps:.1f})")
    
    finally:
        # Clean up
        cap.release()
        out.release()
        cv2.destroyAllWindows()
        pbar.close()
    
    # Calculate and display statistics
    if processing_times:
        avg_processing_time = np.mean(processing_times)
        avg_fps = 1.0 / avg_processing_time
        
        print(f"\\nProcessing completed!")
        print(f"Frames processed: {frame_count}")
        print(f"Average processing time per frame: {avg_processing_time:.3f} seconds")
        print(f"Average processing FPS: {avg_fps:.1f}")
        print(f"Output video saved: {output_path}")
        
        return {
            'frames_processed': frame_count,
            'avg_processing_time': avg_processing_time,
            'avg_fps': avg_fps,
            'output_path': output_path
        }
    else:
        print("No frames were processed")
        return None


In [ ]:
# Process the video
print("Starting video segmentation...")
print("Press 'q' to stop processing early if preview is enabled")

try:
    stats = process_video_with_segmentation(
        input_video_path,
        output_video_path,
        model,
        conf_threshold=confidence_threshold,
        show_preview=show_preview,
        max_frames=max_frames
    )
    
    if stats:
        print(f"\\n✅ Video processing completed successfully!")
        print(f"📁 Output saved to: {stats['output_path']}")
        
except Exception as e:
    print(f"❌ Error processing video: {e}")


In [ ]:
# Optional: Display sample frames from the processed video
def display_sample_frames(video_path, num_samples=4):
    """
    Display sample frames from the processed video
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"Cannot open video: {video_path}")
        return
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sample_indices = np.linspace(0, total_frames-1, num_samples, dtype=int)
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Sample Frames from Segmented Video', fontsize=16)
    
    for i, frame_idx in enumerate(sample_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        
        if ret:
            # Convert BGR to RGB for matplotlib
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            row = i // 2
            col = i % 2
            axes[row, col].imshow(frame_rgb)
            axes[row, col].set_title(f'Frame {frame_idx}')
            axes[row, col].axis('off')
    
    cap.release()
    plt.tight_layout()
    plt.show()

# Display sample frames if output video exists
if os.path.exists(output_video_path):
    print("\\nDisplaying sample frames from the segmented video:")
    display_sample_frames(output_video_path)
else:
    print(f"Output video not found: {output_video_path}")


In [ ]:
# Advanced: Batch processing multiple videos
def batch_process_videos(input_folder, output_folder, model, conf_threshold=0.5):
    """
    Process multiple videos in a folder
    """
    input_path = Path(input_folder)
    output_path = Path(output_folder)
    output_path.mkdir(exist_ok=True)
    
    # Supported video extensions
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv']
    
    # Find all video files
    video_files = []
    for ext in video_extensions:
        video_files.extend(input_path.glob(f'*{ext}'))
        video_files.extend(input_path.glob(f'*{ext.upper()}'))
    
    if not video_files:
        print(f"No video files found in {input_folder}")
        return
    
    print(f"Found {len(video_files)} video files to process")
    
    results = []
    
    for video_file in video_files:
        print(f"\\nProcessing: {video_file.name}")
        
        output_file = output_path / f"segmented_{video_file.name}"
        
        try:
            stats = process_video_with_segmentation(
                str(video_file),
                str(output_file),
                model,
                conf_threshold=conf_threshold,
                show_preview=False  # Disable preview for batch processing
            )
            
            if stats:
                results.append({
                    'input_file': video_file.name,
                    'output_file': output_file.name,
                    'status': 'success',
                    **stats
                })
            
        except Exception as e:
            print(f"Error processing {video_file.name}: {e}")
            results.append({
                'input_file': video_file.name,
                'status': 'failed',
                'error': str(e)
            })
    
    # Print summary
    print(f"\\n{'='*50}")
    print("BATCH PROCESSING SUMMARY")
    print(f"{'='*50}")
    
    successful = [r for r in results if r['status'] == 'success']
    failed = [r for r in results if r['status'] == 'failed']
    
    print(f"Total videos: {len(video_files)}")
    print(f"Successful: {len(successful)}")
    print(f"Failed: {len(failed)}")
    
    if successful:
        avg_fps = np.mean([r['avg_fps'] for r in successful])
        print(f"Average processing FPS: {avg_fps:.1f}")
    
    return results

# Example usage (uncomment to use):
# batch_results = batch_process_videos(
#     input_folder="input_videos",
#     output_folder="output_videos",
#     model=model,
#     conf_threshold=0.5
# )


In [ ]:
# Performance optimization tips and model comparison
print("\\n🚀 PERFORMANCE OPTIMIZATION TIPS:")
print("1. Use smaller models (yolo11n-seg) for faster processing")
print("2. Use larger models (yolo11x-seg) for better accuracy")
print("3. Reduce input resolution for faster processing")
print("4. Increase confidence threshold to reduce false positives")
print("5. Use GPU acceleration if available")
print("6. Process videos in batches for efficiency")

print("\\n📊 MODEL COMPARISON:")
models_info = {
    'yolo11n-seg': {'size': '~6MB', 'speed': 'Fastest', 'accuracy': 'Good'},
    'yolo11s-seg': {'size': '~22MB', 'speed': 'Fast', 'accuracy': 'Better'},
    'yolo11m-seg': {'size': '~50MB', 'speed': 'Medium', 'accuracy': 'Very Good'},
    'yolo11l-seg': {'size': '~110MB', 'speed': 'Slow', 'accuracy': 'Excellent'},
    'yolo11x-seg': {'size': '~220MB', 'speed': 'Slowest', 'accuracy': 'Best'}
}

for model_name, info in models_info.items():
    print(f"{model_name:12} | Size: {info['size']:8} | Speed: {info['speed']:8} | Accuracy: {info['accuracy']}")

print("\\n💡 Choose the model based on your requirements:")
print("   - Real-time processing: yolo11n-seg or yolo11s-seg")
print("   - High accuracy needed: yolo11l-seg or yolo11x-seg")
print("   - Balanced performance: yolo11m-seg")
